In [196]:
import argparse
from model.TSPN import Transparent_Signal_Processing_Network
from trainer.trainer_basic import Basic_plmodel

import torch
from pytorch_lightning import seed_everything
from configs.config import parse_arguments,config_network
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
parser = argparse.ArgumentParser(description='TSPN')
# parser.add_argument('--config_dir', type=str, default='configs/a_031_HUST/config_basic.yaml',help='The directory of the configuration file')
parser.add_argument('--config_dir', type=str, default='configs/a_010_SEU/config_basic.yaml',help='The directory of the configuration file')
# 适用于jupyter
meta_args = parser.parse_known_args()[0]
config_dir = meta_args.config_dir
configs,args,path,name = parse_arguments(config_dir, 0)
signal_processing_modules, feature_extractor_modules = config_network(configs,args)
MODEL_DICT = {'TSPN': lambda args: Transparent_Signal_Processing_Network(signal_processing_modules, feature_extractor_modules,args)}
model_plain = MODEL_DICT[args.model](args)
model = Basic_plmodel(model_plain, args)
state_dict = torch.load("./save/test/model_tson_seu_20hz.ckpt")
# state_dict = torch.load("./save/test/model_tson_hust_20hz.ckpt")
model.load_state_dict(state_dict['state_dict'])
print(model)

Running experiment: model_TSPNtime01-23-12-35_datasetSEU_010_Basic_it0
# build signal processing layers
# build feature extractor layers
# build classifier
Basic_plmodel(
  (network): Transparent_Signal_Processing_Network(
    (signal_processing_layers): ModuleList(
      (0): SignalProcessingLayer(
        (norm): InstanceNorm1d(2, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (weight_connection): Linear(in_features=2, out_features=12, bias=True)
        (signal_processing_modules): SignalProcessingModuleDict(
          (HT): HilbertTransform()
          (LNO): Laplace_neural_operator()
          (I): Identity()
        )
        (skip_connection): Linear(in_features=2, out_features=12, bias=True)
      )
      (1-3): 3 x SignalProcessingLayer(
        (norm): InstanceNorm1d(12, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (weight_connection): Linear(in_features=12, out_features=12, bias=True)
        (signal_processing_modules)

C:\Users\CCSLab\AppData\Local\Temp\ipykernel_4156\143114081.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("./save/test/model_tson_seu_20hz.ckp

In [197]:
def plot_weight(weight, name):
    plt.imshow(weight, cmap='Oranges')
    plt.xticks([])
    plt.yticks([])
    plt.colorbar()
    plt.savefig('save/figure/weight/'+ f'{name}.svg')
    plt.clf()
weight_sol = model.network.signal_processing_layers[3].weight_connection.weight
weight_sol = weight_sol.detach().cpu().numpy()
weight_fol = model.network.feature_extractor_layers.weight_connection.weight
weight_fol = weight_fol.detach().cpu().numpy()
# plot_weight(weight_sol, 'weight_sol')
# plot_weight(weight_fol, 'weight_fol')

In [198]:
from model.TSPN import SignalProcessingLayer, FeatureExtractorlayer, Classifier
import torch.nn as nn
def get_all_layers(module, layers=None):
    """
    递归收集所有子模块。
    
    Args:
        module (nn.Module): 要遍历的主模块。
        layers (list, optional): 用于存储子模块的列表。默认为 None。
        
    Returns:
        list: 所有子模块的列表。
    """
    if layers is None:
        layers = []
    for child in module.children():
        # 如果子模块没有更深层的子模块，则直接添加
        if not list(child.children()):
            layers.append(child)
        else:
            # 否则，递归调用自身
            get_all_layers(child, layers)
    return layers

def save_linear_weights_to_excel(model, filename):
    """
    保存模型中所有 Linear 层的权重到 Excel 文件中，每个 Linear 层的权重保存在不同的工作表中。
    在每个工作表中，第一列为操作符名称，第一行为通道编号。
    
    Args:
        model (nn.Module): 要处理的模型。
        filename (str): 保存的 Excel 文件名（包括路径）。
    """
    # 获取所有子模块
    layers = get_all_layers(model)
    # 筛选出 Linear 层
    linear_layers = [layer for layer in layers if isinstance(layer, nn.Linear)]
    
    if not linear_layers:
        print("模型中没有找到 Linear 层。")
        return
    
    # 使用 Pandas 的 ExcelWriter 保存到 Excel
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        for idx, layer in enumerate(linear_layers):
            # 尝试获取模块的名称和路径
            module_name = None
            for name, mod in model.named_modules():
                if mod is layer:
                    module_name = name
                    break
            if module_name is None:
                module_name = f'Linear_{idx+1}'
            
            # Excel 工作表名称限制为 31 个字符
            sheet_name = module_name[:31]
            
            # Determine the context of the Linear layer
            parent_module = None
            for name, mod in model.named_modules():
                if layer in list(mod.children()):
                    parent_module = mod
                    break
            
            operator_names = []
            if isinstance(parent_module, SignalProcessingLayer):
                # 获取 signal_processing_modules 的名称
                modules = list(parent_module.signal_processing_modules.keys())
                num_modules = len(modules)
                out_features = layer.weight.data.size(0)
                channels_per_module = out_features // num_modules
                # 如果无法均分，分配多一些给前面的模块
                counts = [channels_per_module + 1 if i < out_features % num_modules else channels_per_module for i in range(num_modules)]
                for module, count in zip(modules, counts):
                    operator_names.extend([module] * count)
            elif isinstance(parent_module, FeatureExtractorlayer):
                # 获取 feature_extractor_modules 的名称
                modules = list(parent_module.feature_extractor_modules.keys())
                num_modules = len(modules)
                out_features = layer.weight.data.size(0)
                print(layer.weight.data.shape)
                channels_per_module = out_features // num_modules
                counts = [channels_per_module + 1 if i < out_features % num_modules else channels_per_module for i in range(num_modules)]
                for module, count in zip(modules, counts):
                    operator_names.extend([module] * count)
            elif isinstance(parent_module, Classifier):
                # 使用 feature1, feature2, ... 作为操作符名称
                out_features = layer.weight.data.size(0)
                operator_names = [f'feature{i}' for i in range(out_features)]
            else:
                # 其他 Linear 层，使用默认名称
                out_features = layer.weight.data.size(0)
                operator_names = [f'Linear_{idx}_out_{i}' for i in range(out_features)]
            
            in_features = layer.weight.data.size(1)
            weight = layer.weight.data.cpu().numpy()
            
            # 检查 operator_names 是否与 out_features 匹配
            if len(operator_names) != weight.shape[0]:
                print(f"警告: 在模块 '{module_name}' 中，操作符名称数量 ({len(operator_names)}) 与输出特征数量 ({weight.shape[0]}) 不匹配。将使用默认名称。")
                operator_names = [f'out_{i}' for i in range(weight.shape[0])]
            
            # 创建 DataFrame
            df = pd.DataFrame(weight)
            # 添加操作符名称作为第一列
            df.insert(0, 'Operator', operator_names)
            # 创建通道编号列表
            channel_numbers = [f'Ch_{i}' for i in range(weight.shape[1])]
            # 设置第一行为通道编号
            df.columns = ['Operator'] + channel_numbers
            
            # 将 DataFrame 写入 Excel 工作表
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"所有 Linear 层的权重已保存到 {filename}")
save_linear_weights_to_excel(model.network, 'save/figure/weight/weight.xlsx')

torch.Size([12, 12])
所有 Linear 层的权重已保存到 save/figure/weight/weight.xlsx


In [199]:

def get_top_k_weights(idx, layer,k = 10,weight_flag = 'weight_connection'):
    
    """
    获取权重的前 k 个值 get_top_k_weights
    """
    
    print('layer:',idx)
    if weight_flag == 'weight_connection':
        weight = layer.weight_connection.weight # .detach().cpu().numpy()
    elif weight_flag == 'skip_connection':
        layer.skip_connection.weight.data = F.softmax((5.0 / 0.2) *  # 0.09 / 0.2
                                                    torch.abs(layer.skip_connection.weight.data), dim=0)
        weight = layer.skip_connection.weight
    elif weight_flag == 'clf':
        weight = layer.weight
    else:
        return None
    # 计算权重的绝对值
    abs_weight = torch.abs(weight)

    # 将权重矩阵扁平化
    flat_abs_weights = abs_weight.flatten()

    # 使用 topk 方法找到扁平化权重中最大的 k 个元素及其索引
    values, flat_indices = flat_abs_weights.topk(k, largest=True)

    # 如果需要，将扁平化后的索引转换回原始矩阵的二维索引
    row_indices, col_indices = np.unravel_index(flat_indices.cpu().numpy(), abs_weight.shape)
    result_list = []

    # 在循环中将每个元素的信息添加到列表中
    for i in range(k):
        if values[i].item() <= 0.1:
            continue
        if col_indices[i]+1 not in [2,3,5,6,10,12]:
            continue
        result_list.append((row_indices[i]+1, col_indices[i]+1))
    return result_list


import networkx as nx

def signal_weight(model,k=20,weight_flag = 'weight_connection'):
    """
    每一层的权重提取信号权重提取 signal_weight
    
    """
    layer_top_weight = {}
    for idx, layer in enumerate(model.signal_processing_layers):
        result_list = get_top_k_weights(idx, layer,k=k,weight_flag = weight_flag)
        layer_top_weight[f'layer:{idx}'] = result_list
        print(result_list)
    return layer_top_weight
# connect_weight = signal_weight(model.network, k=20,weight_flag = 'weight_connection')
skip_weight = signal_weight(model.network, k=20,weight_flag = 'skip_connection')


layer: 0
[(2, 2), (4, 2)]
layer: 1
[(12, 5), (5, 2), (1, 6), (4, 10), (7, 10), (3, 12), (1, 12), (6, 3), (3, 3)]
layer: 2
[(9, 5), (1, 12), (2, 10), (5, 2), (10, 6), (2, 12), (7, 3), (10, 2), (1, 6), (11, 10)]
layer: 3
[(1, 3), (2, 2), (2, 12), (4, 5), (3, 6), (9, 6), (4, 10)]


In [200]:
get_top_k_weights(idx = 0, layer=model.network.feature_extractor_layers,k=25, weight_flag ='weight_connection')


layer: 0


[(9, 5),
 (11, 12),
 (6, 6),
 (1, 6),
 (6, 3),
 (9, 2),
 (12, 3),
 (12, 2),
 (9, 12),
 (4, 6),
 (1, 3),
 (5, 10),
 (2, 10),
 (1, 12)]

In [201]:
weight = model.network.clf.clf[0].weight
k = 200
abs_weight = torch.abs(weight)

    # 将权重矩阵扁平化
flat_abs_weights = abs_weight.flatten()

    # 使用 topk 方法找到扁平化权重中最大的 k 个元素及其索引
values, flat_indices = flat_abs_weights.topk(k, largest=True)

    # 如果需要，将扁平化后的索引转换回原始矩阵的二维索引
row_indices, col_indices = np.unravel_index(flat_indices.cpu().numpy(), abs_weight.shape)
result_list = []

    # 在循环中将每个元素的信息添加到列表中
alist = [1,4,5,6,9,11,12]

for idx in alist:
    j = 0
    for i in range(k):
        if j > 2:
            break
        if values[i].item() <= 0.1:
            continue
        if col_indices[i]// 13 + 1 == idx :
            print(col_indices[i]// 13 + 1, col_indices[i] - 13 * (col_indices[i]// 13)+1)
            j = j + 1
    

        

1 13
1 13
1 13
4 8
4 6
4 6
5 10
5 2
5 2
6 6
6 7
6 6
9 1
9 11
11 2
11 2
11 2
